# SQL AI Agent - Database Logging Examples

This notebook demonstrates how to set up database logging for the SQL AI Agent with both **PostgreSQL** and **DuckDB**.

## What You'll Learn

1. 📘 **PostgreSQL Setup** - Initialize log schema and use database logging
2. 📗 **DuckDB Setup** - Initialize log schema for in-memory or persistent storage
3. 🔍 **Query Logs** - Analyze logs using SQL queries
4. 📊 **Best Practices** - Tips for production use

## Benefits of Database Logging

- ✅ **Structured Storage** - All logs in queryable database tables
- ✅ **Easy Analytics** - Use SQL for log analysis
- ✅ **Performance** - Optimized indexes for fast queries
- ✅ **Integration** - Works with existing database infrastructure

---

## Setup and Imports

First, let's import all the necessary modules.

In [2]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [7]:
import ibis
import pandas as pd
import logging
import json
from datetime import datetime

# SQL AI Agent imports
from sql_ai_agent.log_database import (
    init_postgres_log_schema,
    init_duckdb_log_schema,
    DatabaseLogHandler,
    verify_log_table,
    drop_log_table
)
from sql_ai_agent.logger import SQLAgentLogger
from sql_ai_agent.SqlAgent import SqlAgent
from sql_ai_agent.data import get_ibis_connection
print("✅ All imports successful!")

✅ All imports successful!


---

# Part 1: PostgreSQL Database Logging

## 📘 PostgreSQL Setup

We'll demonstrate:
1. Connecting to PostgreSQL
2. Initializing the log schema
3. Setting up database logging
4. Using SqlAgent with database logging
5. Querying the logs

### Step 1: Connect to PostgreSQL

**Prerequisites:**
- PostgreSQL server running
- Database created
- Appropriate credentials

Update the connection parameters below with your settings:

In [9]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con_postgres = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

### Step 2: Initialize PostgreSQL Log Schema

This creates the `sql_agent_logs` table with:
- Optimized schema for log storage
- 6 indexes for fast queries
- JSONB field for flexible metadata

In [ ]:
# Initialize log table
init_postgres_log_schema(
    con=con_postgres,
    table_name="sql_agent_logs",
    schema="public"
)

print("\n✅ PostgreSQL log table initialized!")
print("   Table: public.sql_agent_logs")
print("   Indexes: 6 created for query performance")

### Step 3: Verify Table Creation

Let's verify the table was created successfully:

In [10]:
# Verify table
stats = verify_log_table(postgres_con, "sql_agent_logs", "postgres")

print("📊 Table Statistics:")
print(f"   Table exists: {stats['exists']}")
print(f"   Total logs: {stats['row_count']}")

if stats['row_count'] > 0:
    print(f"   Earliest log: {stats['earliest_log']}")
    print(f"   Latest log: {stats['latest_log']}")
    print(f"   Log levels: {stats['log_levels']}")

NameError: name 'postgres_con' is not defined

### Step 4: Set Up Database Logging Handler

Configure Python's logging system to write to the database:

In [ ]:
# Create logger
logger = logging.getLogger('sql_ai_agent')
logger.setLevel(logging.INFO)
logger.handlers.clear()  # Clear any existing handlers

# Add database handler
db_handler = DatabaseLogHandler(
    con=postgres_con,
    table_name="sql_agent_logs",
    db_type="postgres",
    schema="public",
    level=logging.INFO
)
logger.addHandler(db_handler)

# Also add console handler for visibility
console_handler = logging.StreamHandler()
console_handler.setFormatter(
    logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
)
logger.addHandler(console_handler)

print("✅ Database logging configured!")
print("   Logs will be written to: public.sql_agent_logs")

### Step 5: Test Database Logging

Let's write some test logs to verify everything works:

In [ ]:
# Create SQLAgentLogger adapter for structured logging
agent_logger = SQLAgentLogger(logger, extra={'session_id': 'notebook-postgres-demo'})

# Write test logs
agent_logger.info(
    "PostgreSQL database logging initialized",
    extra={
        'operation_type': 'initialization',
        'database_type': 'postgres',
        'table_name': 'sql_agent_logs'
    }
)

agent_logger.info(
    "Test log entry",
    extra={
        'operation_type': 'test',
        'test_field': 'example_value',
        'test_number': 42
    }
)

print("\n✅ Test logs written to database!")

### Step 6: Query PostgreSQL Logs

Let's verify the logs were written and query them:

In [ ]:
# Query recent logs
query = """
SELECT 
    id,
    timestamp,
    level,
    message,
    operation_type,
    session_id
FROM sql_agent_logs
ORDER BY timestamp DESC
LIMIT 5;
"""

recent_logs = postgres_con.raw_sql(query).to_pandas()

print("📋 Recent logs from PostgreSQL:")
print(recent_logs.to_string(index=False))

### Step 7: Create Sample Data Table (for SqlAgent demo)

Let's create a sample table to demonstrate SqlAgent with database logging:

In [ ]:
# Create sample employees table
sample_data = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'name': ['Alice Johnson', 'Bob Smith', 'Charlie Brown', 'Diana Prince', 'Eve Davis'],
    'age': [28, 35, 42, 31, 29],
    'department': ['Engineering', 'Sales', 'Engineering', 'HR', 'Sales'],
    'salary': [95000, 75000, 110000, 65000, 80000]
})

# Create table in PostgreSQL
try:
    # Drop if exists and recreate
    postgres_con.raw_sql("DROP TABLE IF EXISTS employees;")
    postgres_con.raw_sql("""
        CREATE TABLE employees (
            id INT PRIMARY KEY,
            name VARCHAR(100),
            age INT,
            department VARCHAR(50),
            salary INT
        );
    """)
    
    # Insert data
    for _, row in sample_data.iterrows():
        postgres_con.raw_sql(f"""
            INSERT INTO employees VALUES 
            ({row['id']}, '{row['name']}', {row['age']}, '{row['department']}', {row['salary']});
        """)
    
    print("✅ Sample 'employees' table created in PostgreSQL")
    print(f"   Rows: {len(sample_data)}")
    print(f"\n{sample_data.to_string(index=False)}")
except Exception as e:
    print(f"Note: {e}")
    print("This is optional - you can use any existing table.")

### Step 8 (Optional): Use SqlAgent with PostgreSQL Database Logging

**Note:** This step requires an API key. Skip if you don't have one.

In [ ]:
# Optional: Use SqlAgent with database logging
# Uncomment and provide your API key to run this section

# API_KEY = "your-openai-api-key-here"  # Replace with your key
#
# agent = SqlAgent(
#     api_key=API_KEY,
#     base_url="https://api.openai.com/v1",
#     model="gpt-4o-mini",
#     con=postgres_con,
#     tbl_name="employees",
#     fallback=False,
#     fallback_model="gpt-4o-mini",
#     enable_logging=True,
#     log_level="INFO",
#     log_file=None,  # Database only
#     log_to_console=False,
#     memory=True,
#     memory_size=5
# )
#
# print("✅ SqlAgent created with PostgreSQL database logging")
#
# # Execute a query
# result = agent.ask_question("How many employees are there?", verbose=True)

print("⏭️ Skipped: SqlAgent demo (requires API key)")
print("   Uncomment the code above and add your API key to test SqlAgent.")

### Step 9: Analyze PostgreSQL Logs

Let's run some analytical queries on the logs:

In [ ]:
# Query 1: Count logs by level
query1 = """
SELECT level, COUNT(*) as count
FROM sql_agent_logs
GROUP BY level
ORDER BY count DESC;
"""

print("📊 Logs by Level:")
result1 = postgres_con.raw_sql(query1).to_pandas()
print(result1.to_string(index=False))

# Query 2: Count logs by operation type
query2 = """
SELECT 
    operation_type,
    COUNT(*) as count
FROM sql_agent_logs
WHERE operation_type IS NOT NULL
GROUP BY operation_type
ORDER BY count DESC;
"""

print("\n📊 Logs by Operation Type:")
result2 = postgres_con.raw_sql(query2).to_pandas()
print(result2.to_string(index=False))

# Query 3: Table statistics
stats = verify_log_table(postgres_con, "sql_agent_logs", "postgres")
print(f"\n📊 Total Logs in PostgreSQL: {stats['row_count']}")

---

# Part 2: DuckDB Database Logging

## 📗 DuckDB Setup

DuckDB is perfect for:
- **Development & Testing** - In-memory database
- **Portable Logging** - Single file database
- **Fast Analytics** - Optimized for analytical queries

We'll demonstrate:
1. Connecting to DuckDB (in-memory)
2. Initializing the log schema
3. Setting up database logging
4. Testing and querying logs

### Step 1: Connect to DuckDB

DuckDB can run in-memory or with a persistent file:

In [ ]:
# Option 1: In-memory DuckDB (data lost when connection closes)
duckdb_con = ibis.duckdb.connect()
print("✅ Connected to DuckDB (in-memory)")

# Option 2: Persistent DuckDB file (uncomment to use)
# duckdb_con = ibis.duckdb.connect("logs/agent_logs.db")
# print("✅ Connected to DuckDB: logs/agent_logs.db")

### Step 2: Initialize DuckDB Log Schema

This creates the `sql_agent_logs` table with optimized schema:

In [ ]:
# Initialize log table
init_duckdb_log_schema(
    con=duckdb_con,
    table_name="sql_agent_logs"
)

print("\n✅ DuckDB log table initialized!")
print("   Table: sql_agent_logs")
print("   Auto-increment sequence created")

### Step 3: Verify DuckDB Table Creation

In [ ]:
# Verify table
stats = verify_log_table(duckdb_con, "sql_agent_logs", "duckdb")

print("📊 Table Statistics:")
print(f"   Table exists: {stats['exists']}")
print(f"   Total logs: {stats['row_count']}")

### Step 4: Set Up DuckDB Logging Handler

In [ ]:
# Create separate logger for DuckDB
duckdb_logger = logging.getLogger('sql_ai_agent_duckdb')
duckdb_logger.setLevel(logging.INFO)
duckdb_logger.handlers.clear()

# Add database handler
duckdb_db_handler = DatabaseLogHandler(
    con=duckdb_con,
    table_name="sql_agent_logs",
    db_type="duckdb",
    level=logging.INFO
)
duckdb_logger.addHandler(duckdb_db_handler)

# Add console handler
duckdb_console_handler = logging.StreamHandler()
duckdb_console_handler.setFormatter(
    logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
)
duckdb_logger.addHandler(duckdb_console_handler)

print("✅ DuckDB logging configured!")
print("   Logs will be written to: sql_agent_logs")

### Step 5: Test DuckDB Logging

In [ ]:
# Create SQLAgentLogger adapter
duckdb_agent_logger = SQLAgentLogger(
    duckdb_logger, 
    extra={'session_id': 'notebook-duckdb-demo'}
)

# Write test logs
duckdb_agent_logger.info(
    "DuckDB database logging initialized",
    extra={
        'operation_type': 'initialization',
        'database_type': 'duckdb',
        'table_name': 'sql_agent_logs'
    }
)

duckdb_agent_logger.info(
    "Test log entry for DuckDB",
    extra={
        'operation_type': 'test',
        'test_field': 'duckdb_value',
        'test_number': 100
    }
)

duckdb_agent_logger.warning(
    "Example warning log",
    extra={
        'operation_type': 'validation',
        'warning_type': 'example'
    }
)

print("\n✅ Test logs written to DuckDB!")

### Step 6: Query DuckDB Logs

In [ ]:
# Query recent logs
query = """
SELECT 
    id,
    timestamp,
    level,
    message,
    operation_type,
    session_id
FROM sql_agent_logs
ORDER BY timestamp DESC
LIMIT 5;
"""

recent_logs = duckdb_con.con.execute(query).df()

print("📋 Recent logs from DuckDB:")
print(recent_logs.to_string(index=False))

### Step 7: Create Sample Data in DuckDB

In [ ]:
# Create sample employees table in DuckDB
sample_data = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'age': [28, 35, 42, 31, 29, 38],
    'department': ['Engineering', 'Sales', 'Engineering', 'HR', 'Sales', 'Engineering'],
    'salary': [95000, 75000, 110000, 65000, 80000, 105000]
})

# Create table in DuckDB
duckdb_con.create_table('employees', sample_data, overwrite=True)

print("✅ Sample 'employees' table created in DuckDB")
print(f"   Rows: {len(sample_data)}")
print(f"\n{sample_data.to_string(index=False)}")

### Step 8: Analyze DuckDB Logs

In [ ]:
# Query 1: Count logs by level
query1 = """
SELECT level, COUNT(*) as count
FROM sql_agent_logs
GROUP BY level
ORDER BY count DESC;
"""

print("📊 Logs by Level:")
result1 = duckdb_con.con.execute(query1).df()
print(result1.to_string(index=False))

# Query 2: Count logs by operation type
query2 = """
SELECT 
    operation_type,
    COUNT(*) as count
FROM sql_agent_logs
WHERE operation_type IS NOT NULL
GROUP BY operation_type
ORDER BY count DESC;
"""

print("\n📊 Logs by Operation Type:")
result2 = duckdb_con.con.execute(query2).df()
print(result2.to_string(index=False))

# Query 3: Table statistics
stats = verify_log_table(duckdb_con, "sql_agent_logs", "duckdb")
print(f"\n📊 Total Logs in DuckDB: {stats['row_count']}")

### Step 9: Advanced DuckDB Log Queries

DuckDB's JSON support makes it easy to query metadata:

In [ ]:
# Query JSON fields in DuckDB
query = """
SELECT 
    timestamp,
    level,
    message,
    json_extract(extra_fields, '$.operation_type') as operation_type,
    json_extract(extra_fields, '$.database_type') as database_type
FROM sql_agent_logs
WHERE extra_fields IS NOT NULL
ORDER BY timestamp DESC
LIMIT 5;
"""

print("📊 Detailed Log View (with JSON extraction):")
result = duckdb_con.con.execute(query).df()
print(result.to_string(index=False))

---

# Part 3: Comparison & Best Practices

## PostgreSQL vs DuckDB for Logging

| Feature | PostgreSQL | DuckDB |
|---------|-----------|--------|
| **Use Case** | Production, shared logging | Development, testing, portable |
| **Setup** | Requires server | In-memory or single file |
| **Performance** | Excellent for concurrent writes | Excellent for analytical queries |
| **JSON Support** | JSONB with GIN indexes | JSON with extraction functions |
| **Persistence** | Always persistent | Optional (in-memory or file) |
| **Sharing** | Multi-user database | Single process (file mode) |
| **Indexes** | 6 optimized indexes | No manual indexes needed |

## Best Practices

### 1. Choose the Right Database

**Use PostgreSQL when:**
- Running in production
- Multiple services/agents logging to same database
- Need concurrent write access
- Integration with existing PostgreSQL infrastructure

**Use DuckDB when:**
- Development and testing
- Single agent/service
- Need portable log storage
- Performing analytical queries on logs

### 2. Log Level Configuration

```python
# Production: INFO level (reduces log volume)
db_handler = DatabaseLogHandler(con, "sql_agent_logs", "postgres", level=logging.INFO)

# Development: DEBUG level (detailed logging)
db_handler = DatabaseLogHandler(con, "sql_agent_logs", "duckdb", level=logging.DEBUG)
```

### 3. Table Maintenance (PostgreSQL)

```sql
-- Periodic maintenance
VACUUM ANALYZE sql_agent_logs;

-- Archive old logs (keep last 30 days)
DELETE FROM sql_agent_logs WHERE timestamp < NOW() - INTERVAL '30 days';
```

### 4. Query Performance Tips

```sql
-- Use indexes: filter by timestamp, level, session_id
SELECT * FROM sql_agent_logs 
WHERE timestamp > NOW() - INTERVAL '1 hour'
  AND level = 'ERROR';

-- Query JSONB efficiently (PostgreSQL)
SELECT * FROM sql_agent_logs
WHERE extra_fields @> '{"operation_type": "query_result"}'::jsonb;

-- Extract JSON fields (DuckDB)
SELECT json_extract(extra_fields, '$.operation_type') as op_type
FROM sql_agent_logs;
```

---

# Summary

## What We Covered

✅ **PostgreSQL Setup**
- Connected to PostgreSQL
- Initialized log schema with 6 indexes
- Configured database logging
- Queried and analyzed logs

✅ **DuckDB Setup**
- Connected to DuckDB (in-memory)
- Initialized log schema
- Configured database logging
- Queried logs with JSON extraction

✅ **Best Practices**
- Choosing the right database
- Log level configuration
- Query optimization
- Table maintenance

## Next Steps

1. **Integrate with SqlAgent**
   - Add your API key
   - Uncomment SqlAgent demo code
   - Run queries and see logs populate

2. **Build Dashboards**
   - Query logs for analytics
   - Track token usage
   - Monitor error rates

3. **Production Deployment**
   - Use PostgreSQL for production
   - Set up log rotation/archival
   - Configure monitoring alerts

## Resources

- 📘 [DATABASE_LOGGING.md](../docs/DATABASE_LOGGING.md) - Complete documentation
- 📄 [database_logging_examples.py](../examples/database_logging_examples.py) - Python examples
- 🔧 [log_database.py](../sql_ai_agent/log_database.py) - Source code

---

**Happy Logging! 🎉**

## Cleanup (Optional)

Run this cell to drop the log tables if you want to start fresh:

In [ ]:
# Uncomment to drop tables

# # Drop PostgreSQL table
# try:
#     drop_log_table(postgres_con, "sql_agent_logs", "postgres", schema="public", confirm=True)
# except Exception as e:
#     print(f"PostgreSQL: {e}")

# # Drop DuckDB table
# try:
#     drop_log_table(duckdb_con, "sql_agent_logs", "duckdb", confirm=True)
# except Exception as e:
#     print(f"DuckDB: {e}")

print("⏭️ Cleanup skipped - uncomment code above to drop tables")